# PRISM · Minute Change Lab
## End-to-end execution on free Colab

Select **Runtime → Change runtime type → T4 GPU** when available. The browser app and generation work without a GPU; Qwen inference and QLoRA require one. No trained model or validated accuracy is preloaded. Human annotations cannot be replaced by invented scores.

1. Upload the project ZIP downloaded from the site.
2. Generate or import images, validate and split by original.
3. Export blind tasks and obtain four independent human annotations.
4. Import annotations; calibrate on training only.
5. Run actual baseline, prompt selection, QLoRA and evaluation.

Free Colab can disconnect and may not allocate a GPU. Save artifacts to Drive. Start with two pairs to check memory. P4 uses four images and is more expensive than the two-image prompts.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, subprocess, sys, json
uploaded = files.upload()  # Choose prism-project.zip
archive = next(name for name in uploaded if name.endswith('.zip'))
root = Path('/content/prism_iqa'); root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        destination=(root/member.filename).resolve()
        if not destination.is_relative_to(root.resolve()): raise ValueError('Unsafe archive path')
    z.extractall(root)
os.chdir(root)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','tesseract-ocr','fonts-dejavu-core'],check=True)
print('Project ready at', root)


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
else: print('Generation/annotation/evaluation still work. Select a GPU before inference or training.')
def run(*args):
    subprocess.run([sys.executable,*args],check=True)
run('-m','unittest','discover','-s','tests','-v')


## 1. Generate releases
The default generator makes **20 authored development originals × 25 variants = 500 pairs**, including unchanged controls. It is a runnable pipeline exercise, not a representative research benchmark. Use the real-scan route below for the final corpus. Existing original source images are not automatically uploaded to Colab.


In [ ]:
DATA_ROOT = Path('data/generated')
PAIRS = DATA_ROOT/'pairs.jsonl'
if not PAIRS.exists():
    run('-m','prism','generate','--output',str(DATA_ROOT),'--originals','20','--variants','25')
run('-m','prism','audit',str(PAIRS),'--root',str(DATA_ROOT),'--output','reports/assets.json')


### Optional: process a real scan
Upload one of your original images. OCR and translation are drafts. Inspect the JSON, correct columns/bounding boxes and target text, and set `approved: true` only after review. The renderer rejects unapproved baselines. For a same-language diagnostic pilot, copy verified source text into `text_translated` and use `target_lang: en`; label it as such.


In [ ]:
# uploaded = files.upload()
# scan = next(iter(uploaded))
# run('scripts/prepare_scan.py','ocr',scan,'data/draft.json')
# draft=json.loads(Path('data/draft.json').read_text())
# draft['target_lang']='fr'  # or 'es'
# Path('data/draft.json').write_text(json.dumps(draft,ensure_ascii=False,indent=2))
# run('scripts/prepare_scan.py','translate','data/draft.json','data/translated.json')
# files.download('data/translated.json')
# Review and upload the corrected approved JSON before running:
# run('scripts/prepare_scan.py','generate','data/translated.json','data/scan_001')


## 2. Grouped split and blinded annotation
Do this before model selection. All variants of an original stay together. Exact source hashes are grouped. The private key never goes to raters. For a representative study, additionally group template families/near duplicates and diversify source content.


In [ ]:
SPLIT = Path('data/split.jsonl')
run('-m','prism','split',str(PAIRS),'--output',str(SPLIT))
run('-m','prism','tasks',str(SPLIT),'--root',str(DATA_ROOT),'--output','data/annotation','--calibration','300')
import shutil
shutil.make_archive('blind-review-bundle','zip','data/annotation/blinded')
files.download('blind-review-bundle.zip')


Use the website's Annotation studio: import `tasks.jsonl` and select its image folder. Each of four people enters a unique stable ID and exports JSONL. Collect actual ratings; do not copy one person's scores to four IDs. The test set and validation set both require human scores. Stop here until annotations exist.


In [ ]:
# Upload the four exported annotation JSONL files.
uploaded = files.upload()
annotation_files=[name for name in uploaded if name.endswith('.jsonl')]
assert annotation_files, 'Upload independent annotation files before continuing.'
Path('data/all_annotations.jsonl').write_text('\n'.join(Path(f).read_text() for f in annotation_files))
run('-m','prism','consensus','data/all_annotations.jsonl','--key','data/annotation/private_assignment_key.json','--output','data/consensus.jsonl')
run('-m','prism','calibrate',str(SPLIT),'data/consensus.jsonl','--labelled-output','data/labelled.jsonl')


## 3. Genuine zero-shot baseline and validation selection
The runner uses only images and the fixed rubric, never seeded severity or human targets. Quality sees Original then Release. The separate detection probe sees localized baseline then Release. Evaluate baseline test performance once; do not use it to choose prompts or tune the model.


In [ ]:
assert torch.cuda.is_available(), 'A CUDA runtime is required for Qwen.'
run('-m','prism','infer',str(SPLIT),'--root',str(DATA_ROOT),'--split','test','--output','outputs/baseline.jsonl','--probe')
run('scripts/select_prompt.py',str(SPLIT),'data/consensus.jsonl','--root',str(DATA_ROOT))
selection=json.loads(Path('outputs/prompt_selection/selection.json').read_text())['selected']
print('Frozen validation selection:',selection)


## 4. QLoRA and best-checkpoint inference
NF4 4-bit; all-linear rank 16/alpha 32; FP16 for T4; batch 1 and accumulation 8; assistant-only loss; cosine schedule; 2 epochs; best validation loss checkpoint. If memory runs out, restart the runtime and resume with a memory-feasible protocol configuration chosen on validation. Do not silently downscale the held-out test.


In [ ]:
run('-m','prism','train','data/labelled.jsonl','--root',str(DATA_ROOT),'--prompt',selection['prompt'],'--resolution',str(selection['resolution']),'--epochs','2')
run('-m','prism','infer',str(SPLIT),'--root',str(DATA_ROOT),'--adapter','outputs/qlora/best','--prompt',selection['prompt'],'--resolution',str(selection['resolution']),'--output','outputs/finetuned.jsonl','--probe')


## 5. Human agreement and minute-change sensitivity
Reports include exclusions, grouped bootstrap intervals, per-metric MAE/RMSE/PLCC/SROCC, within-original ranking and detection audits. A threshold remains unresolved when there are too few samples or negative controls. This is preferable to an unsupported accuracy claim.


In [ ]:
for name in ['baseline','finetuned']:
    run('-m','prism','evaluate',str(SPLIT),'data/consensus.jsonl',f'outputs/{name}.jsonl','--output',f'reports/{name}.json')
    report=json.loads(Path(f'reports/{name}.json').read_text())
    print(name, report.get('overall'), report.get('ranking'))
shutil.make_archive('prism-results','zip','reports')
files.download('prism-results.zip')
files.download('outputs/finetuned.jsonl')  # import in the website's Evaluation page


## Save for the next free Colab session
The following mount asks for your Google Drive permission. Run it only if you want to store your project/results there. No credentials belong in source files.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copytree('outputs','/content/drive/MyDrive/PRISM/outputs',dirs_exist_ok=True)
# shutil.copytree('data/annotation','/content/drive/MyDrive/PRISM/annotation',dirs_exist_ok=True)
